# BirdCLEF2026 - Research and External Resources

This notebook documents research findings, relevant resources, and code to download external data for the BirdCLEF2026 Challenge.

## Challenge Overview
- **Goal**: Identify 234 wildlife species (birds, amphibians, mammals, reptiles, insects) from audio recordings
- **Data**: 1-minute ogg audio files at 32kHz from Pantanal wetlands, Brazil
- **Evaluation**: Predictions for 5-second segments in test soundscapes

## Why This Challenge Matters

The Pantanal wetlands are one of the world's most diverse ecosystems, home to 650+ bird species. Traditional biodiversity monitoring is expensive and logistically challenging due to seasonal flooding and remote locations. This competition advances passive acoustic monitoring (PAM) technology that can help researchers:

1. **Scale monitoring** - Deploy 1000+ autonomous recording units across the wetland
2. **Track biodiversity** - Monitor how species respond to climate change and conservation efforts
3. **Enable rapid response** - Detect presence of endangered species or invasive threats

The challenge is significant because:
- Audio is recorded in **field conditions** with background noise, weather, and overlapping species
- The **domain shift** between training data (clean Xeno-canto recordings) and test data (field recordings) is substantial
- With **234 classes** and significant class imbalance, this is a challenging multi-label problem

## Key Research Findings

### Winning Approaches from BirdCLEF 2024/2025

We analyzed recent BirdCLEF competitions to understand what techniques work best:

#### 1. 2nd Place BirdCLEF+ 2025 (ROC AUC: 0.928)

**Why this approach matters:**
- **Domain shift** between clean training recordings and noisy field recordings is the biggest challenge
- **Transfer learning** from in-domain data helps bridge this gap
- **Model distillation** allows semi-supervised learning from unlabeled field data
- **Ensembling** multiple models provides robustness

Key components:
- Transfer learning with in-domain pre-training
- Semi-supervised learning via model distillation
- Post-processing and model ensembling
- Code: https://github.com/VSydorskyy/BirdCLEF_2025_2nd_place

#### 2. Bird Whisperer (InterSpeech 2024)

**Why Whisper works well:**
- Whisper was trained on 680k hours of diverse audio including non-speech sounds
- The encoder captures general acoustic patterns that transfer to bird calls
- Fine-tuning is more effective than using frozen features
- **15% improvement** over baseline demonstrates the power of transfer learning

- Code: https://github.com/umer-sheikh/bird-whisperer

#### 3. EfficientNet Ensemble (Top 100 BirdCLEF 2025)

**Why EfficientNet is a solid baseline:**
- EfficientNet architecture is optimized for efficiency and accuracy
- Pretrained ImageNet weights provide useful feature extractors
- Mel spectrograms can be treated as images, leveraging CNN architecture
- Fast training makes experimentation and iteration easier

- Code: https://github.com/ambruhsia/BirdCLEF-2025

## Decision Rationale: Why This Approach?

Based on our research, we've chosen an **ensemble approach** combining:

| Component | Choice | Rationale |
|-----------|--------|----------|
| **Primary Model** | EfficientNet-B0 | Fast training, good baseline performance, proven in BirdCLEF competitions |
| **Secondary Model** | Whisper (tiny/base) | Transfer learning from diverse audio, handles domain shift |
| **Feature Format** | Mel Spectrograms | Standard for bioacoustic classification, treats audio as 2D images |
| **Ensemble** | Weighted Average | Simple but effective combination strategy |

### Why EfficientNet-B0 First?

1. **Speed**: B0 is the smallest EfficientNet variant, enabling fast iteration
2. **Memory**: Fits in 24GB VRAM with batch size 32
3. **Performance**: Provides ~75-80% of larger model performance with 10x faster training
4. **Proven**: Consistently used in top BirdCLEF solutions

### Why Whisper as Secondary?

1. **Pretrained knowledge**: Trained on 680k hours of diverse audio
2. **Domain adaptation**: Can be fine-tuned to capture bird-specific patterns
3. **Complementary**: Different architecture provides ensemble diversity
4. **Research support**: Bird Whisperer showed 15% improvement

### Why Not PaSST or Other State-of-Art?

1. **VRAM constraints**: PaSST requires 6GB+ VRAM alone
2. **Complexity**: More complex to implement and tune
3. **Diminishing returns**: For this competition, simpler methods may suffice
4. **Time constraints**: EfficientNet provides faster iteration

### Recommended Approaches

| Approach | Pros | Cons | When to Use |
|---------|------|------|------------|
| EfficientNet-B0/B1 | Fast training, good baseline | Less capacity | First model, quick iteration |
| Whisper (tiny/base) | Transfer learning, handles domain shift | Slower inference | Domain shift is significant |
| PaSST | Audio-pretrained, state-of-art | Requires more VRAM | Have sufficient compute |
| Ensemble | Best performance | Complex | Final submission |

## External Resources

### Pre-trained Models

We use two main sources for pretrained models:

1. **timm (PyTorch Image Models)**: Provides EfficientNet, ConvNeXt, and other CNN architectures pretrained on ImageNet
2. **HuggingFace Transformers**: Provides Whisper and other audio pretrained models

**Why ImageNet pretrained models work:**
- Mel spectrograms are 2D images with time on x-axis and frequency on y-axis
- ImageNet features capture low-level patterns (edges, textures) that transfer to spectrogram patterns
- This is a well-established technique in audio classification

In [ ]:
# PyTorch Image Models (timm) - EfficientNet, ConvNeXt, ViT
# https://github.com/huggingface/pytorch-image-models
import timm

# List available efficientnet models
efficientnet_models = [m for m in timm.list_models('*efficientnet*') if 'b0' in m or 'b1' in m]
print("Available EfficientNet models:", efficientnet_models[:10])

In [ ]:
# HuggingFace Transformers - Whisper for audio
# https://huggingface.co/models?pipeline_tag=automatic-speech-recognition
from transformers import AutoModelForAudioClassification, AutoProcessor

# Whisper models available:
# - openai/whisper-tiny (39M params, fastest)
# - openai/whisper-base (74M params)
# - openai/whisper-small (244M params)
# - openai/whisper-medium (769M params, most capacity)
# 
# For this challenge, we recommend:
# - whisper-tiny for initial experiments
# - whisper-base for final ensemble (if time permits)
print("Whisper models: whisper-tiny, whisper-base, whisper-small, whisper-medium")

### Audio Processing Libraries

**Why these libraries?**

- **librosa**: The gold standard for audio analysis in Python, provides all necessary transformations
- **torchaudio**: PyTorch integration for GPU-accelerated audio processing
- **audiomentations**: Easy-to-use audio augmentation pipeline with proven techniques
- **soundfile**: Fast, reliable audio I/O for ogg format

In [ ]:
# Core libraries for audio processing
import librosa  # Mel spectrograms, audio loading
import torchaudio  # PyTorch audio
import audiomentations  # Data augmentation
import soundfile as sf  # Audio I/O

# Check versions
print(f"librosa: {librosa.__version__}")
print(f"torchaudio: {torchaudio.__version__}")

### External Datasets

**Important: Competition Rules**

The training data already includes Xeno-canto and iNaturalist recordings. Per competition rules:
- Do NOT re-scrape these sources
- Limit burden on their servers

However, these external resources can help with **transfer learning**:


In [ ]:
# External resources for transfer learning (do NOT use for training labels):

# 1. BirdNET - https://birdnet.cornell.edu/
#    - Pre-trained models for bird species identification
#    - Can be used for pseudo-labeling or transfer learning
#    - Trained on millions of bird recordings

# 2. DCASE (Detection and Classification of Acoustic Scenes and Events)
#    - https://dcase.community/
#    - Acoustic scene classification techniques
#    - Audio event detection methods

# 3. PANNs (Pre-trained Audio Neural Networks)
#    - https://github.com/yinkalario/General-Purpose-Audio-Recognition
#    - Large-scale pre-trained audio models
#    - AudioSet pretrained models available

# Note: These are for model weights/knowledge, not for obtaining additional training labels
print("External resources documented - see comments for usage")

## Code to Download External Models

While timm will automatically download weights, having a function to pre-download models ensures reproducibility.

In [ ]:
import os
from pathlib import Path

# Create models directory
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

# Download EfficientNet-B0 pretrained weights (if not already available via timm)
# timm will automatically download weights on first use - no action needed

# Download Whisper model
def download_whisper_model(model_name="tiny"):
    """Download and cache Whisper model for transfer learning.
    
    Why Whisper?
    - Trained on 680k hours of diverse audio
    - Includes non-speech sounds that may resemble bird calls
    - Fine-tuning adapts the encoder to bird-specific patterns
    
    Args:
        model_name: 'tiny', 'base', 'small', or 'medium'
    
    Returns:
        model, processor: Whisper model and processor
    """
    from transformers import WhisperForAudioClassification, WhisperProcessor
    
    model_path = MODELS_DIR / f"whisper-{model_name}"
    model_path.mkdir(exist_ok=True)
    
    # This will download the model from HuggingFace
    processor = WhisperProcessor.from_pretrained(f"openai/whisper-{model_name}")
    model = WhisperForAudioClassification.from_pretrained(f"openai/whisper-{model_name}")
    
    return model, processor

# Usage:
# model, processor = download_whisper_model("tiny")
# Note: For full training, we'll use whisper as feature extractor, not fine-tune the entire model
print("Model download functions defined. Uncomment to download.")

## Pre-trained Model Benchmarks

This comparison helps us choose models based on our compute constraints:

| Model | Parameters | VRAM (batch=32) | Speed | Recommended For |
|-------|-----------|-----------------|-------|-----------------|
| EfficientNet-B0 | 5.3M | 2GB | Fast | First model, baseline |
| EfficientNet-B1 | 7.8M | 3GB | Fast | Better accuracy if time permits |
| EfficientNet-B2 | 9.2M | 4GB | Medium | Best accuracy/compute tradeoff |
| Whisper-tiny | 39M | 1GB | Fast | Transfer learning starter |
| Whisper-base | 74M | 2GB | Medium | Main transfer model |
| PaSST | 81M | 6GB | Slow | If compute not a concern |

**Our choice**: EfficientNet-B0 for initial model (fast iteration), Whisper-base for ensemble diversity

In [ ]:
# Compare model sizes and expected performance
model_info = {
    "EfficientNet-B0": {"params": "5.3M", "vram_estimate": "2GB", "speed": "Fast", "use_case": "Baseline model"},
    "EfficientNet-B1": {"params": "7.8M", "vram_estimate": "3GB", "speed": "Fast", "use_case": "More capacity"},
    "EfficientNet-B2": {"params": "9.2M", "vram_estimate": "4GB", "speed": "Medium", "use_case": "Best accuracy"},
    "Whisper-tiny": {"params": "39M", "vram_estimate": "1GB", "speed": "Fast", "use_case": "Transfer learning"},
    "Whisper-base": {"params": "74M", "vram_estimate": "2GB", "speed": "Medium", "use_case": "Main transfer model"},
    "PaSST": {"params": "81M", "vram_estimate": "6GB", "speed": "Slow", "use_case": "State-of-art"},
}

import pandas as pd

pd.DataFrame(model_info).T

## Environment Setup

Let's verify our compute environment is ready.

In [ ]:
# Check GPU availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
# Verify compute is sufficient for our approach
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb >= 24:
        print("\n✓ VRAM sufficient for EfficientNet-B0 + Whisper ensemble")
    elif vram_gb >= 16:
        print("\n✓ VRAM sufficient for EfficientNet-B0 (may need smaller batch)")
    else:
        print("\n⚠ Consider using EfficientNet-B0 only or reducing batch size")

## Recommended Dependencies

These packages are required for the complete pipeline:

In [ ]:
# Install required packages
required_packages = """
# Core ML
torch>=2.0.0
torchvision
torchaudio
timm
transformers

# Audio processing
librosa
soundfile
audiomentations

# Data handling
pandas
numpy
scikit-learn

# Visualization
matplotlib
seaborn

# Misc
tqdm
pillow
"""
print(required_packages)

## Summary: Recommended Approach

Based on our research, here's the complete approach:

### Feature Extraction
- **Mel spectrograms** with 128 mel bands, 5-second windows at 32kHz
- This converts 1D audio to 2D representation suitable for CNNs
- Parameters chosen based on standard bioacoustic practice

### Model Architecture
1. **Primary**: EfficientNet-B0 pretrained on ImageNet
   - Fast training, good baseline
   - Replace final layer for 234-class classification

2. **Secondary**: Whisper-base as feature extractor
   - Transfer learning from diverse audio
   - Helps with domain shift

### Ensemble Strategy
- Average predictions from both models
- Simple but effective

### Data Augmentation
- **Time stretching**: Simulates speed variations in bird calls
- **Pitch shifting**: Simulates different bird sizes/ages
- **Background noise**: Improves robustness to field conditions
- **Time shifting**: Improves temporal robustness

### Training Strategy
- Use both train_audio (file-level) and train_soundscapes (segment-level)
- Binary cross-entropy loss for multi-label classification
- AdamW optimizer with cosine annealing learning rate
- Early stopping based on validation AUC

### Why This Will Work

1. **EfficientNet** provides a solid baseline proven in previous BirdCLEF competitions
2. **Whisper** addresses the domain shift between clean training data and noisy field recordings
3. **Ensemble** provides robustness and improved generalization
4. **Data augmentation** improves generalization to unseen field conditions

This approach balances accuracy, training time, and implementation complexity.